# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tqdm.notebook import tqdm
import joblib

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy()
        df['hour'] = df['timestamp'].dt.hour
        df['dayofweek'] = df['timestamp'].dt.dayofweek
        df = df.drop(['timestamp'], axis=1)
        return df

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column):
        self.target_column = target_column
        self.encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        self.categorical_columns = None
        self.feature_names = None
    
    def fit(self, X, y=None):
        self.categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        if self.target_column in self.categorical_columns:
            self.categorical_columns.remove(self.target_column)
        
        if self.categorical_columns:
            self.encoder.fit(X[self.categorical_columns])
            self.feature_names = self.encoder.get_feature_names_out(self.categorical_columns)
        
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        y = X_copy[self.target_column]
        X_features = X_copy.drop(self.target_column, axis=1)
        
        if self.categorical_columns:
            encoded_array = self.encoder.transform(X_features[self.categorical_columns])
            encoded_df = pd.DataFrame(
                encoded_array, 
                columns=self.feature_names,
                index=X_features.index
            )
            
            X_features = X_features.drop(self.categorical_columns, axis=1) # удаляем категориальные признаки
            
            X_transformed = pd.concat([X_features, encoded_df], axis=1)
        else:
            X_transformed = X_features
        
        return X_transformed, y

In [4]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state
    
    def fit(self, X, y):
        return self
    
    def transform(self, X, y):
        X_train_valid, X_test, y_train_valid, y_test = train_test_split(
            X, y, 
            test_size=self.test_size, 
            random_state=self.random_state, 
            stratify=y
        )
        
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_valid, y_train_valid,
            test_size=self.test_size, 
            random_state=self.random_state,
            stratify=y_train_valid
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_results_df = None
        self.best_overall_model = None
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        results = []
        
        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            
            print(f"\nEstimator: {model_name}")
            
            total_combinations = self._calculate_total_combinations(grid.param_grid)
            print(f"Total parameter combinations: {total_combinations}")
            
            pbar = tqdm(total=total_combinations, desc=f"Fitting {model_name}")
            original_fit = grid.fit
            
            def custom_fit(X, y=None, **fit_params):
                original_run_search = grid._run_search
                
                def run_search_with_progress(evaluate_candidates):
                    def evaluate_with_progress(candidate_params):
                        result = evaluate_candidates(candidate_params)
                        pbar.update(len(candidate_params))
                        return result
                    return original_run_search(evaluate_with_progress)
                
                grid._run_search = run_search_with_progress
                result = original_fit(X, y, **fit_params)
                return result
                
            grid.fit = custom_fit
            
            try:
                grid.fit(X_train, y_train)
            finally:
                grid.fit = original_fit
                pbar.close()
            
            best_estimator = grid.best_estimator_
            best_params = grid.best_params_
            best_train_score = grid.best_score_
            
            y_pred_valid = best_estimator.predict(X_valid)
            valid_score = accuracy_score(y_valid, y_pred_valid)
            
            results.append({
                'model': model_name,
                'params': best_params,
                'train_score': best_train_score,
                'valid_score': valid_score,
                'estimator': best_estimator
            })
            
            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")
        
        self.best_results_df = pd.DataFrame(results)
        best_idx = self.best_results_df['valid_score'].idxmax()
        self.best_overall_model = self.best_results_df.loc[best_idx, 'model']
        
        print(f"\nClassifier with best validation set accuracy: {self.best_overall_model}")
        
        return self.best_overall_model
    
    def _calculate_total_combinations(self, param_grid):
        total = 0
        for params in param_grid:
            if isinstance(params, dict):
                combinations = 1
                for key, values in params.items():
                    combinations *= len(values)
                total += combinations
            else:
                # Если param_grid это список словарей
                combinations = 1
                for key, values in params.items():
                    combinations *= len(values)
                total += combinations
        return total
    
    def best_results(self):
        if self.best_results_df is None:
            raise ValueError("Must call choose() method first")
        return self.best_results_df[['model', 'params', 'valid_score']].reset_index(drop=True)
    
    def get_best_estimator(self, model_name=None):
        if self.best_results_df is None:
            raise ValueError("Must call choose() method first")
        if model_name is None:
            model_name = self.best_overall_model
        best_row = self.best_results_df[self.best_results_df['model'] == model_name].iloc[0]
        return best_row['estimator']

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [6]:
class Finalize:
    def __init__(self, model):
        self.model = model
    
    def final_score(self, X_train, y_train, X_test, y_test):
        self.model.fit(X_train, y_train)
        score = self.model.score(X_test, y_test)
        print(f'Accuracy of the final model is: {score}')
        return score
    
    def save_model(self, path):
        joblib.dump(self.model, path)

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv('../data/checker_submits.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['dayofweek'] = df['timestamp'].dt.dayofweek

df.head()

,uid,labname,numTrials,timestamp,dayofweek
0,user_4,project1,1,2020-04-17 05:19:02.744528,4
1,user_4,project1,2,2020-04-17 05:22:45.549397,4
2,user_4,project1,3,2020-04-17 05:34:24.422370,4
3,user_4,project1,4,2020-04-17 05:43:27.773992,4
4,user_4,project1,5,2020-04-17 05:46:32.275104,4


In [8]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()), 
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

In [9]:
data = preprocessing.fit_transform(df)
X, y = data

splitter = TrainValidationTest(test_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.transform(X, y)

In [10]:
svm_params = [{
    'kernel': ['linear', 'rbf', 'sigmoid'], 
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}]

tree_params = [{
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 20, 21, 22, 25, 30],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]

rf_params = [{
    'n_estimators': [10, 30, 50, 100],
    'criterion': ['gini', 'entropy'],
    'max_depth': [15, 20, 22, 25, 28, 30],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]

gs_svm = GridSearchCV(
    estimator=SVC(), 
    param_grid=svm_params, 
    scoring='accuracy', 
    cv=2,
    verbose=0
)

gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(), 
    param_grid=tree_params, 
    scoring='accuracy', 
    cv=2,
    verbose=0
)

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(), 
    param_grid=rf_params, 
    scoring='accuracy', 
    cv=2, 
    verbose=0
)

grids = [gs_svm, gs_tree, gs_rf]
grids_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

selection = ModelSelection(grids, grids_dict)

best_model_name = selection.choose(X_train, y_train, X_valid, y_valid)

model_df = selection.best_results()
print(model_df)


Estimator: SVM
Total parameter combinations: 72


Fitting SVM:   0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.768
Validation set accuracy score for best params: 0.885

Estimator: Decision Tree
Total parameter combinations: 28


Fitting Decision Tree:   0%|          | 0/28 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.844

Estimator: Random Forest
Total parameter combinations: 96


Fitting Random Forest:   0%|          | 0/96 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 30, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.858
Validation set accuracy score for best params: 0.893

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': 'balanced', 'criterion': 'gin...   
2  Random Forest  {'class_weight': None, 'criterion': 'entropy',...   

   valid_score  
0     0.885185  
1     0.844444  
2     0.892593  


In [11]:
best_estimator = selection.get_best_estimator()

final = Finalize(best_estimator)
test_accuracy = final.final_score(
    pd.concat([X_train, X_valid]), 
    pd.concat([y_train, y_valid]), 
    X_test, 
    y_test
)

Accuracy of the final model is: 0.9349112426035503


In [12]:
final.save_model('../data/model_04.pkl')